<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>청킹 전략(Chunking Strategy)</strong>에 대해 학습합니다.
>긴 문서를 효과적으로 분할하는 청킹 기법과 파라미터의 영향을 학습해봅시다.

# 청킹 (Chunking)
> 긴 문서를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">적절한 크기의 조각</mark>으로 분할하여 검색 정확도와 LLM 처리 효율을 높입니다.

긴 문서를 그대로 LLM에 넣지 못하는 데에는 두 가지 이유가 있습니다. 첫째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM 토큰 제한</mark>으로 LLM은 한 번에 처리할 수 있는 텍스트 길이(컨텍스트 윈도우)에 상한이 있어서 수백 페이지짜리 PDF를 통째로 프롬프트에 넣는 것은 불가능하거나, 가능하더라도 비용이 매우 큽니다. 둘째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">검색 정확도 향상</mark>을 위해 문서를 작은 단위로 쪼개두면, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">질문과 가장 관련 있는 청크만 선별</mark>하여 LLM에 전달할 수 있습니다. 즉, 청킹은 토큰 제한 안에서 검색 품질을 최대화하기 위한 필수 전처리 단계입니다. 청크가 너무 작으면 문맥이 끊기고, 너무 크면 LLM에 불필요한 정보까지 전달됩니다.</br>

이 과정에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">토큰 제한</mark>(LLM이 한 번에 처리할 수 있는 최대 토큰 수, 즉 컨텍스트 윈도우 개념)과 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문서 로더</mark>(PDF 등 파일을 `Document` 객체로 변환하는 방법, Ch.4-1_002 참고)에 대한 이해가 필요합니다.

## RecursiveCharacterTextSplitter
> LangChain의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">재귀적 문자 분할기</mark>로, 구분자 우선순위에 따라 의미 단위로 분할합니다.

In [ ]:
# TODO 1: 재귀적 텍스트 분할기를 chunk_size=500, chunk_overlap=50, 구분자 우선순위 ["\n\n", "\n", ".", " "]로 초기화하고, 문서를 분할하여 청크 개수와 크기 통계를 출력해봅시다.

from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 청크 최대 크기
    chunk_overlap=50,    # 청크 간 겹침
    separators=["\n\n", "\n", ".", " "]  # 분할 우선순위
)

chunks = splitter.split_documents(documents)
print(f"원본: {len(documents)}개 → 청크: {len(chunks)}개")

# 청크 크기 확인
sizes = [len(c.page_content) for c in chunks]
print(f"청크 크기 — 평균: {sum(sizes)/len(sizes):.0f}, 최소: {min(sizes)}, 최대: {max(sizes)}")

## 주요 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th>영향</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>chunk_size</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청크 최대 문자 수</mark></td><td>작을수록 정밀, 클수록 문맥 보존</td></tr>
    <tr><td style="text-align:center"><code>chunk_overlap</code></td><td>인접 청크 간 겹침</td><td>문맥 단절 방지</td></tr>
    <tr><td style="text-align:center"><code>separators</code></td><td>분할 기준 문자</td><td>의미 단위 보존</td></tr>
  </tbody>
</table>

## 청크 크기의 트레이드오프

In [ ]:
# TODO 2: chunk_size를 200, 500, 1000으로 변경하며 재귀적 텍스트 분할기(chunk_overlap=50)로 문서를 분할하고, 각 설정별 청크 수를 비교해봅시다.

for size in [200, 500, 1000]:
    s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    c = s.split_documents(documents)
    print(f"chunk_size={size:4d} → 청크 수: {len(c):3d}개")

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">chunk_size</th>
      <th style="text-align:center">청크 수</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">200</td><td style="text-align:center">87개</td></tr>
    <tr><td style="text-align:center">500</td><td style="text-align:center">42개</td></tr>
    <tr><td style="text-align:center">1000</td><td style="text-align:center">23개</td></tr>
  </tbody>
</table>

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">chunk_size</th>
      <th>장점</th>
      <th>단점</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">작음 (200)</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정밀한 검색</mark></td><td>문맥 손실</td></tr>
    <tr><td style="text-align:center">중간 (500)</td><td>균형 잡힌 성능</td><td>-</td></tr>
    <tr><td style="text-align:center">큼 (1000)</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">풍부한 문맥</mark></td><td>검색 정밀도 저하</td></tr>
  </tbody>
</table>

💡chunk_overlap의 역할
> overlap이 없으면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문장이 중간에 잘릴</mark> 수 있습니다.
> 일반적으로 chunk_size의 10~20%를 overlap으로 설정합니다.

💡청킹이 RAG 성능을 좌우한다
> 같은 모델, 같은 임베딩이라도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청킹 전략</mark>에 따라 검색 품질이 크게 달라집니다.
> 문서 특성에 맞는 chunk_size와 separator를 실험적으로 결정해야 합니다.